# BAC × Grundfos — Week 3 | Hourly heat demand regression

**Author:** Daksh | **Assignment:** Different regression model for each building type | **Dataset:** `data/modeling/hourly_heat.parquet`

### Research question
Can a simple, interpretable regression model predict how much heating + hot-water energy a type of NYC building uses in each hour, using only weather and calendar information?

**Rules:** 17 building types (17 models), **train January–September 2018**, test **October–December 2018**, no historical heating-demand features. The model's target is `heat_total_kwh` and the score is mean CV(RMSE) across the 17 types. Team benchmark: **0.392** (lower is better).

> **Test result from uploaded dataset:** **0.250851** mean CV(RMSE), approximately **36.0% less than the team's benchmark**. Each building type's feature set and Ridge penalty were chosen using February, May and September *within the train partition*, then refit on all Jan–Sep observations. Test months were not used for candidate selection.

### ML lesson
We'll progress through data exploration → feature engineering → model selection → `fit()` → `predict()` → CV(RMSE) → submission. Ordinary linear regression learns one coefficient per input feature; **Ridge** is the same linear form with a penalty on large coefficients. A piecewise `HDH` feature lets the relationship bend without changing the estimator into a tree or neural network.

## 1. Load and inspect the data

**X (inputs):** only weather and calendar information. **y (target):** heating + hot-water demand in kWh. `profile_type` determines *which separate model* is fitted—it isn't a meter-demand feature.

The repository uses `pyarrow` to read Parquet. The pickle fallback below is solely for the analysis workspace when that Parquet engine isn't installed; normal local repo execution uses the `.parquet` file.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

search_at=Path.cwd().resolve()
DATA=None
while True:
    for candidate in [search_at / "data/modeling/hourly_heat.parquet", search_at / "hourly_heat.parquet"]:
        if candidate.exists():
            DATA=candidate
            break
    if DATA or search_at==search_at.parent:
        break
    search_at=search_at.parent
if DATA is None:
    raise FileNotFoundError("Place this notebook inside the cloned repo, or next to hourly_heat.parquet")
ROOT = DATA.parent.parent.parent if DATA.parent.name == "modeling" else DATA.parent
try:
    df = pd.read_parquet(DATA)
except ImportError:
    # File read for this particular hosted analysis; not required in the GitHub repo.
    cached = Path("/mnt/data/hourly_heat_read.pkl")
    if not cached.exists():
        raise ImportError("Install the repo dependencies: pip install -r requirements.txt")
    df = pd.read_pickle(cached)

print("Shape:", df.shape, "| building types:", df.profile_type.nunique())
print("Split counts:")
print(df.data_split.value_counts())
print("Missing values:", int(df.isna().sum().sum()))
display(df[["timestamp", "profile_type", "temp_c", "hdh_18c", "hour_of_day", "heat_total_kwh", "data_split"]].head())

### Understand the physics: heating degree-hours

`hdh_18c = max(0, 18.3 − temp_c)`. When it's colder outside, the heating degree-hour measure rises. A basic regression draws one straight relationship between HDH and heat use. We also include `max(0, HDH - k)` hinge features, so the slope can change at several temperature thresholds.

In [ ]:
example = df.loc[(df.profile_type == "largeoffice") & (df.data_split == "train")]
plt.figure(figsize=(7,4))
plt.scatter(example.hdh_18c, example.heat_total_kwh, s=3, alpha=.12)
plt.title("Large office — training data only")
plt.xlabel("Heating degree-hours (°C · h)")
plt.ylabel("Heating + hot-water demand (kWh/hour)")
plt.tight_layout()
plt.show()

## 2. Define seven regression candidates

All 7 are **linear-in-coefficients regression models**, including OLS and Ridge. We compare basic calendar effects, extra weather variables, HDH squared terms, hour/weekend interactions, and piecewise HDH hinges. `alpha` is the Ridge regularization strength. Greater alpha shrinks coefficients more.

**Why features matter:** a hotel has heating demand at night, whereas an office's heat use may vary sharply between business hours and weekends. Interactions let the model learn those differences separately for each building type.

In [ ]:
TARGET = "heat_total_kwh"
BASE = ["hdh_18c", "year_sin", "year_cos", "is_weekend", "is_holiday", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
WEATHER = ["rel_humidity_pct", "wind_speed_ms", "ghi_wm2", "temp_c_roll24", "temp_c_lag1", "temp_c_lag3", "temp_deviation"]
NONLINEAR = ["hdh_squared", "hdh_x_year_sin", "hdh_x_year_cos", "hdh_x_wind", "hdh_x_solar", "hdh_x_weekend", "hdh_x_hour_sin", "hdh_x_hour_cos"]
HOURS = [f"hour_{i}" for i in range(24)]
DOW = [f"dow_{i}" for i in range(7)]
HDH_HOURS = [f"hdh_hour_{i}" for i in range(24)]
WEEKEND_HOURS = [f"weekend_hour_{i}" for i in range(24)]
HDH_WEEKEND_HOURS = [f"hdh_weekend_hour_{i}" for i in range(24)]
HDH_DOW = [f"hdh_dow_{i}" for i in range(7)]
HINGE_KNOTS = [2,4,6,8,10,12,15,18,21,24,28,32]
HINGES = [f"hdh_hinge_{k}" for k in HINGE_KNOTS]
FULL_CORE = ["hdh_18c", "year_sin", "year_cos", "year_sin2", "year_cos2", "is_weekend", "is_holiday"]
HINGE_CORE = ["hdh_18c", "year_sin", "year_cos", "is_weekend", "is_holiday"]

# Seven linear-model variants; none are trees or neural networks.
# Candidate columns and regularization are fixed BEFORE consulting the test set.
CANDIDATES: dict[str,tuple[list[str],float]] = {
 "ols_basic": (BASE, 0),
 "ridge_weather_100": (BASE+WEATHER, 100),
 "ridge_smooth_100": (BASE+WEATHER+NONLINEAR+["hour_sin2","hour_cos2","year_sin2","year_cos2"], 100),
 "ridge_full_100": (FULL_CORE+HOURS+DOW+WEATHER+NONLINEAR+HDH_HOURS+WEEKEND_HOURS+HDH_WEEKEND_HOURS+HDH_DOW, 100),
 "ridge_hinge_100": (HINGE_CORE+HOURS+DOW+WEATHER+HINGES+NONLINEAR+HDH_HOURS, 100),
 "ridge_hinge_full_100": (HINGE_CORE+HOURS+DOW+WEATHER+HINGES+NONLINEAR+HDH_HOURS+WEEKEND_HOURS+HDH_WEEKEND_HOURS, 100),
 "ridge_hinge_full_1000": (HINGE_CORE+HOURS+DOW+WEATHER+HINGES+NONLINEAR+HDH_HOURS+WEEKEND_HOURS+HDH_WEEKEND_HOURS, 1000),
}


In [ ]:
def prepare_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Make numeric features from WEATHER and CALENDAR only, with no target leakage."""
    required = ["timestamp","timestamp_local","profile_type","data_split",TARGET,
                "hdh_18c","hour_of_day","day_of_week","month","is_weekend","is_holiday",
                "rel_humidity_pct","wind_speed_ms","ghi_wm2","temp_c","temp_c_roll24",
                "temp_c_lag1","temp_c_lag3"]
    missing = sorted(set(required)-set(raw.columns))
    if missing: raise ValueError(f"Missing required dataset columns: {missing}")
    if raw[["timestamp","profile_type"]].duplicated().any():
        raise ValueError("Duplicate building-type / timestamp keys")
    if set(raw.data_split.unique()) != {"train","test"}:
        raise ValueError("Unexpected data split values")

    d=raw.copy()
    d["timestamp"]=pd.to_datetime(d["timestamp"])
    d["timestamp_local"]=pd.to_datetime(d["timestamp_local"])
    h=d.hour_of_day.astype(int).to_numpy()
    weekday=d.day_of_week.astype(int).to_numpy()
    day=d.timestamp_local.dt.dayofyear.to_numpy()
    hdh=d.hdh_18c.to_numpy(dtype=float)
    weekend=d.is_weekend.to_numpy(dtype=bool)
    f={}
    f["hdh_squared"]=hdh**2
    f["year_sin"]=np.sin(2*np.pi*day/365)
    f["year_cos"]=np.cos(2*np.pi*day/365)
    f["year_sin2"]=np.sin(4*np.pi*day/365)
    f["year_cos2"]=np.cos(4*np.pi*day/365)
    f["hour_sin"]=np.sin(2*np.pi*h/24)
    f["hour_cos"]=np.cos(2*np.pi*h/24)
    f["hour_sin2"]=np.sin(4*np.pi*h/24)
    f["hour_cos2"]=np.cos(4*np.pi*h/24)
    f["dow_sin"]=np.sin(2*np.pi*weekday/7)
    f["dow_cos"]=np.cos(2*np.pi*weekday/7)
    f["hdh_x_year_sin"]=hdh*f["year_sin"]
    f["hdh_x_year_cos"]=hdh*f["year_cos"]
    f["hdh_x_wind"]=hdh*d.wind_speed_ms.to_numpy(dtype=float)
    f["hdh_x_solar"]=hdh*d.ghi_wm2.to_numpy(dtype=float)
    f["hdh_x_weekend"]=hdh*weekend
    f["hdh_x_hour_sin"]=hdh*f["hour_sin"]
    f["hdh_x_hour_cos"]=hdh*f["hour_cos"]
    f["temp_deviation"]=d.temp_c-d.temp_c_roll24
    for hr in range(24):
        f[f"hour_{hr}"]=(h==hr).astype(int)
        f[f"hdh_hour_{hr}"]=hdh*(h==hr)
        f[f"weekend_hour_{hr}"]=((h==hr)&weekend).astype(int)
        f[f"hdh_weekend_hour_{hr}"]=hdh*((h==hr)&weekend)
    for w in range(7):
        f[f"dow_{w}"]=(weekday==w).astype(int)
        f[f"hdh_dow_{w}"]=hdh*(weekday==w)
    for knot in HINGE_KNOTS:
        f[f"hdh_hinge_{knot}"]=np.maximum(0,hdh-knot)
    return pd.concat([d,pd.DataFrame(f,index=d.index)],axis=1)

features = prepare_features(df)
print("Engineered feature count:", features.shape[1] - df.shape[1])
print("Candidate inputs:", {name: len(cols) for name, (cols, alpha) in CANDIDATES.items()})

## 3. Learn the model fitting and scoring functions

`fit(X_train, y_train)` estimates coefficients from the training rows. `predict(X)` applies those coefficients to unseen hours. A pipeline makes sure missing-value imputation and scaling are **learned only from training rows**, never the validation or test set.

The official metric normalizes root-mean-square error by the mean actual demand **within each building type**. We score each type separately and take their arithmetic mean; an unusually large office cannot dominate the metric simply because it consumes more energy.

In [ ]:
def new_model(candidate: str):
    """The imputer and scaler are fitted only on the given training rows."""
    _, alpha = CANDIDATES[candidate]
    reg=Ridge(alpha=alpha,solver="lsqr",tol=1e-5,max_iter=2000) if alpha else LinearRegression()
    return make_pipeline(SimpleImputer(strategy="median",keep_empty_features=True), StandardScaler(),reg)

def cvrmse(y, yhat):
    """Exactly the repo score definition for one building profile."""
    y=np.asarray(y,dtype=float)
    yhat=np.asarray(yhat,dtype=float)
    return float(np.sqrt(np.mean((y-yhat)**2))/np.mean(y))

## 4. Choose hyperparameters without using October–December

To select a candidate, we hold out one complete month at a time—**February** (winter), **May** (spring) and **September** (fall)—from the January–September training partition. We train on its other months and evaluate on the held-out month. We pool squared errors across the three months before normalizing, rather than averaging unstable errors from low-demand months individually.

This is **blocked month validation, not a strictly chronological rolling forecast**. It's appropriate for testing weather/calendar patterns because we do not use lagged heating demand. The final held-out test months (Oct–Dec) remain excluded from selection.

In [ ]:
def choose_candidate(training: pd.DataFrame) -> tuple[str,dict[str,float]]:
    """Blocked train-only validation: hold out February, May, September in turn.

    We fit on the OTHER Jan--Sep months, so this is NOT a chronological forecast
    validation. It checks generalization across seasons. No heating-demand lags
    and no test (Oct--Dec) observations enter selection. Pooled RMSE across
    validation months avoids unstable averaging of near-zero summer CV(RMSE).
    """
    scored={}
    for candidate,(cols,_) in CANDIDATES.items():
        residuals=[]
        heldout=[]
        for heldout_month in [2,5,9]:
            fit_rows=training.month!=heldout_month
            val_rows=training.month==heldout_month
            m=new_model(candidate)
            m.fit(training.loc[fit_rows,cols],training.loc[fit_rows,TARGET])
            p=np.maximum(0,m.predict(training.loc[val_rows,cols]))
            y=training.loc[val_rows,TARGET].to_numpy(dtype=float)
            residuals.append(y-p)
            heldout.append(y)
        residuals=np.concatenate(residuals)
        heldout=np.concatenate(heldout)
        scored[candidate]=float(np.sqrt(np.mean(residuals**2))/np.mean(heldout))
    return min(scored,key=scored.get),scored

## 5. Train and predict all 17 building types

The loop is the key learning point: it's one reusable pipeline that produces **17 different fitted coefficient vectors**. We fit the chosen candidate on all Jan–Sep data for each profile and predict its 2,208 test hours, clipping negative predictions to zero.

The test scores in the report are **diagnostics after selection**, not optimization inputs.

In [ ]:
def fit_all(raw:pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame]:
    df=prepare_features(raw)
    predictions=[];results=[]
    for profile in sorted(df.profile_type.unique()):
        one=df[df.profile_type==profile]
        tr=one[one.data_split=="train"]
        te=one[one.data_split=="test"]
        best,validation=choose_candidate(tr)
        cols,_=CANDIDATES[best]
        model=new_model(best)
        model.fit(tr[cols],tr[TARGET])  # final fit uses all Jan--Sep observations
        p=np.maximum(0,model.predict(te[cols]))
        y=te[TARGET].to_numpy(dtype=float)
        predictions.append(pd.DataFrame({"timestamp":te.timestamp.to_numpy(),"profile_type":profile,"predicted_kwh":p}))
        results.append({"profile_type":profile,"selected_model":best,
                        "validation_cvrmse":validation[best],
                        "test_cvrmse":cvrmse(y,p),
                        "test_wape":float(np.sum(np.abs(y-p))/np.sum(y)),
                        "test_nmbe":float(np.sum(y-p)/np.sum(y)),
                        "test_mean_kwh":float(np.mean(y)),
                        "test_hours":len(y),"n_features":len(cols)})
    pred=pd.concat(predictions,ignore_index=True).sort_values(["timestamp","profile_type"]).reset_index(drop=True)
    report=pd.DataFrame(results)
    expected=int((df.data_split=="test").sum())
    assert len(pred)==expected==37536, f"Expected 37,536 predictions, got {len(pred)}"
    assert not pred[["timestamp","profile_type"]].duplicated().any()
    assert pred.predicted_kwh.notna().all() and pred.predicted_kwh.ge(0).all()
    return pred,report

In [ ]:
predictions, report = fit_all(df)
print(f"Mean test CV(RMSE): {report.test_cvrmse.mean():.6f}")
print("Team benchmark: 0.392")
print("Percent lower than benchmark:", round((0.392-report.test_cvrmse.mean())/0.392*100,2))
print("Submission rows:", len(predictions))
display(report.sort_values("test_cvrmse",ascending=False).reset_index(drop=True))

## 6. Identify where the model works and fails

The metric averages each profile equally. A poor warehouse score matters as much as a good large-hotel score, regardless of the total kWh. We should report the worst profiles rather than hiding them behind the mean.

In [ ]:
view=report.sort_values("test_cvrmse",ascending=True)
plt.figure(figsize=(10,6))
plt.barh(view.profile_type,view.test_cvrmse)
plt.axvline(.392,ls="--",label="Existing mean benchmark = 0.392")
plt.title("October–December test CV(RMSE) by building type")
plt.xlabel("CV(RMSE), lower is better")
plt.legend()
plt.tight_layout()
plt.show()

# A small actual-versus-predicted example for a single type and 7 days.
type_name="largehotel"
truth=df.loc[(df.data_split=="test")&(df.profile_type==type_name),["timestamp","heat_total_kwh"]]
sel=predictions.loc[predictions.profile_type==type_name,["timestamp","predicted_kwh"]]
joined=truth.merge(sel,on="timestamp").head(168)
plt.figure(figsize=(11,3.5))
plt.plot(joined.timestamp,joined.heat_total_kwh,label="Actual")
plt.plot(joined.timestamp,joined.predicted_kwh,label="Predicted",alpha=.8)
plt.title("Large hotel: first 7 test days")
plt.ylabel("kWh per hour")
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Export the required submission and written report

The team expects a CSV with exactly **`timestamp`, `profile_type`, `predicted_kwh`** for all 37,536 test observations. Then run the official repository scorer (`scripts/score.py`) from the repo root to append your score to the team leaderboard.

In [ ]:
out=ROOT / "submissions" / "daksh_hourly.csv"
report_path=ROOT / "submissions" / "daksh_hourly_report.csv"
out.parent.mkdir(parents=True,exist_ok=True)
predictions.to_csv(out,index=False)
report.to_csv(report_path,index=False)
print("Wrote", out)
print("Wrote", report_path)
print("Next, from repo root: python scripts/score.py hourly submissions/daksh_hourly.csv")

## Results and discussion — submission paragraph

I trained separate linear/Ridge regression models for all 17 building types using heating degree-hours, additional weather variables and calendar features. I tested piecewise heating-degree-hour terms and interactions with hour of day and weekends to capture different thermal demand patterns. I selected feature sets and Ridge regularization for each building type by holding out February, May and September within the January–September training period, then retrained on all training hours and predicted October–December. The resulting mean test **CV(RMSE) was 0.2509**, compared with the team's 0.392 baseline (approximately **36% lower**). More flexible heating-degree-hour relationships and hourly/weekend interactions performed better than basic regression on the training-period validation folds. Large hotels (0.0792) and full-service restaurants (0.0862) had the lowest test errors, while **warehouses were worst (0.8629)**. This suggests the warehouse profile contains demand patterns poorly explained by weather and the calendar alone; we should inspect those errors further rather than claim the model captures every building type accurately.

**Study questions:** Why is `heat_total_kwh` a label but `hdh_18c` a feature? What does `fit` learn? Why use weather lag variables but not heating-demand lag variables? What changes when Ridge alpha increases? Why average CV(RMSE) over profiles instead of over all rows?